# 01 — Data Acquisition

Downloads and caches all datasets for the Autobahn speed-safety analysis.

| # | Dataset | Source | Coverage |
|---|---------|--------|----------|
| 1 | Unfallatlas | opengeodata.nrw.de | Germany, GPS accidents 2016–2024 |
| 2 | Destatis timeseries | destatis.de (Excel) | Germany, Autobahn stats 1979–2021 |
| 3 | CBS OData | opendata.cbs.nl | NL, deaths + vehicle-km |
| 4 | BRON | downloads.rijkswaterstaatdata.nl | NL, GPS accidents 2003–2024 |

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../src')

from autobahn_safety.data_loaders import (
    download_unfallatlas,
    load_unfallatlas,
    download_destatis_timeseries,
    load_destatis_autobahn,
    download_bron,
    load_bron,
    fetch_cbs_odata,
)

load_dotenv()

DATA_RAW = Path('../data/raw')
DATA_RAW.mkdir(parents=True, exist_ok=True)
print('Ready.')

## 1. Germany — Unfallatlas (GPS accident records 2016–2024)

In [ ]:
UNFALLATLAS_DIR = DATA_RAW / 'germany' / 'unfallatlas'
YEARS_DE = list(range(2016, 2025))

for year in YEARS_DE:
    year_dir = UNFALLATLAS_DIR / str(year)
    try:
        download_unfallatlas(year, year_dir)
    except Exception as e:
        print(f'{year}: FAILED — {e}')

In [ ]:
df_unfallatlas = load_unfallatlas(UNFALLATLAS_DIR, YEARS_DE)
print(f'Loaded {len(df_unfallatlas):,} accident records')
print(f'Years: {sorted(df_unfallatlas["year"].unique().tolist())}')
df_unfallatlas.head(3)

## 2. Germany — Destatis accident timeseries (1979–2021)

In [ ]:
DESTATIS_DIR = DATA_RAW / 'germany' / 'destatis'
DESTATIS_DIR.mkdir(parents=True, exist_ok=True)

xlsx_path = download_destatis_timeseries(DESTATIS_DIR)
df_autobahn = load_destatis_autobahn(xlsx_path)
print(f'Destatis Autobahn: {len(df_autobahn)} years ({df_autobahn["year"].min()}–{df_autobahn["year"].max()})')
df_autobahn.tail(5)

## 3. Netherlands — CBS OData (deaths + vehicle-km)

In [ ]:
CBS_DIR = DATA_RAW / 'netherlands' / 'cbs'
CBS_DIR.mkdir(parents=True, exist_ok=True)

# Traffic deaths by province and by mode of transport
for ds_id, name in [('71426ned', 'deaths_by_province'), ('71936ned', 'deaths_by_mode')]:
    out = CBS_DIR / f'{name}.csv'
    if out.exists():
        print(f'{ds_id} ({name}): already cached')
        continue
    df = fetch_cbs_odata(ds_id)
    df.to_csv(out, index=False)
    print(f'{ds_id}: {len(df):,} rows saved')

# Vehicle-km by vehicle type (for accident rate normalisation)
out_vkm = CBS_DIR / 'vehicle_km_by_type.csv'
if not out_vkm.exists():
    df_vkm = fetch_cbs_odata('85395NED')
    df_vkm.to_csv(out_vkm, index=False)
    print(f'85395NED: {len(df_vkm):,} rows saved (vehicle-km)')
else:
    print('85395NED (vehicle-km): already cached')

## 4. Netherlands — BRON (GPS accident records 2003–2024)

Direct ZIP download from `downloads.rijkswaterstaatdata.nl` — no registration required.
Key field: `MAXSNELHD` (speed limit) identifies motorway accidents directly.
`AP3_CODE` codes: `DOD`=fatal, `LET`=injury, `UMS`=material damage only.

In [ ]:
BRON_DIR = DATA_RAW / 'netherlands' / 'bron'
BRON_DIR.mkdir(parents=True, exist_ok=True)

YEARS_NL = list(range(2010, 2025))
for year in YEARS_NL:
    try:
        download_bron(year=year, output_dir=BRON_DIR)
    except Exception as e:
        print(f'{year}: FAILED — {e}')

In [ ]:
bron_years = sorted([
    int(f.stem.split('_')[-1])
    for f in BRON_DIR.glob('bron_accidents_*.csv')
])
print(f'Available BRON years: {bron_years}')

if bron_years:
    df_bron = load_bron(BRON_DIR, bron_years[-1:])  # preview latest year only
    motorway = df_bron[df_bron['MAXSNELHD'].isin(['100', '120', '130'])]
    print(f'Latest year ({bron_years[-1]}): {len(df_bron):,} total, {len(motorway):,} motorway')
    print(f'Severity: {df_bron["AP3_CODE"].value_counts().to_dict()}')
    df_bron.head(3)

## Summary

In [ ]:
print('=== Data acquisition summary ===')
print(f'Unfallatlas : {len(df_unfallatlas):,} records ({YEARS_DE[0]}–{YEARS_DE[-1]})')
print(f'Destatis    : {len(df_autobahn)} years ({df_autobahn["year"].min()}–{df_autobahn["year"].max()})')

for name in ['deaths_by_province', 'deaths_by_mode', 'vehicle_km_by_type']:
    p = CBS_DIR / f'{name}.csv'
    if p.exists():
        n = len(pd.read_csv(p, low_memory=False))
        print(f'CBS {name}: {n:,} rows')

if bron_years:
    total_bron = sum(
        len(pd.read_csv(BRON_DIR / f'bron_accidents_{y}.csv', dtype=str, low_memory=False))
        for y in bron_years
    )
    print(f'BRON        : {total_bron:,} records ({bron_years[0]}–{bron_years[-1]})')
